# Annual Max EVI Export from Pre-computed Landsat Composites

This notebook exports **annual max EVI** using Google's pre-computed Landsat composites
(8-day or 32-day temporal resolution), with two cropland masking strategies:

## Temporal resolution options
- **8-day** (default): `LANDSAT/COMPOSITES/C02/T1_L2_8DAY_EVI` (~46 images/year)
- **32-day** (fallback): `LANDSAT/COMPOSITES/C02/T1_L2_32DAY_EVI` (~12 images/year)

## Cropland masking strategies
1. **Annual mask**: Year-specific cropland mask from GLC-FCS30D. Each year's EVI is masked
   with that year's cropland classification (pixels classified as cropland classes 10, 11, 12, 20).
2. **Stable-footprint mask**: Pixels classified as cropland in the baseline period (2001-2003)
   AND in at least 80% of all years (2001-2022, i.e. >= 18 out of 22 years). This single mask
   is applied uniformly to all years, ensuring EVI trends are measured on land that was
   already established cropland before any deforestation during the study period.

## Note on 8-day max export failures
The 8-day annual max export has been observed to sometimes fail on GEE despite being
less computation than harmonic fitting. Mitigations applied:
- Empty-collection guards via `safe_annual_max`
- `tileScale=4` in `reduceRegions`
- Automated fallback to 32-day if 8-day fails
- Option to split 22-year stack into smaller batches if needed


In [ ]:
# Cell 1: Imports and Authentication
import ee

ee.Authenticate()
ee.Initialize()


In [ ]:
# Cell 2: Parameters
farmSize = ee.Image(
    "users/hadicu06/Postdoc_Bonn/World_Bank_Project_2025/"
    "mode_farmsize_2010_reprojTo4326"
)
tiles = ee.FeatureCollection(
    "users/hadicu06/Postdoc_Bonn/World_Bank_Project_2025/"
    "export_tiles_v1_hasCroplandAndTreeLoss_30N_30S_Africa"
)

params = {
    'targetGrid': farmSize,
    'tiles': tiles,
    'start_date': '2001-01-01',
    'end_date': '2022-12-31',
    'version': 'v260211',
    'gdrive': '',
    # Temporal resolution: '8day' (default) or '32day'
    'temporal_res': '8day',
    # Cropland mask mode: 'annual', 'stable', or 'both'
    'cropland_mask_mode': 'both',
    # Stable mask parameters
    'baseline_years': [2001, 2002, 2003],
    'min_fraction_years': 0.80,  # require cropland in >= 80% of years
    'notRun': True,
}


In [ ]:
# Cell 3: Helper Functions
import math


def safe_annual_max(collection, year, band='EVI'):
    """
    Safely create annual max composite with empty-collection guard.

    Args:
        collection: ee.ImageCollection (e.g. 8-day or 32-day EVI composites)
        year: int
        band: str (default 'EVI')

    Returns:
        ee.Image with a single band named 'evi_max_{year}'
    """
    year = int(year)
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')
    filtered = collection.filterDate(start, end).select(band)

    count = filtered.size()
    default_img = (
        ee.Image.constant(0)
        .rename(band)
        .updateMask(ee.Image.constant(0))
    )

    result = ee.Algorithms.If(
        count.gt(0),
        filtered.max().rename(band),
        default_img,
    )
    return ee.Image(result).rename(f'evi_max_{year}')


def build_annual_cropland_mask(glc30_mosaic):
    """
    Build per-year binary cropland mask from GLC-FCS30D mosaic.

    Args:
        glc30_mosaic: ee.Image with bands 'lc_2001' .. 'lc_2022'

    Returns:
        ee.Image with bands 'lc_2001' .. 'lc_2022', each 1 = cropland, masked 0.
    """
    crop_classes = [10, 11, 12, 20]
    band_names = glc30_mosaic.bandNames()

    def _make_binary(band_name):
        band_name = ee.String(band_name)
        band = glc30_mosaic.select([band_name])
        binary = band.remap(crop_classes, [1] * len(crop_classes), 0)
        return binary.rename([band_name])

    crop_bands = band_names.map(_make_binary)
    cropland_mask = ee.ImageCollection(crop_bands).toBands().selfMask()

    # Remove numeric prefix added by toBands() ('0_lc_2001' -> 'lc_2001')
    original = cropland_mask.bandNames()
    cleaned = original.map(
        lambda name: ee.String(name).split('_').slice(1).join('_')
    )
    return cropland_mask.rename(cleaned)


def build_stable_cropland_mask(
    annual_cropland_mask,
    baseline_years=None,
    min_fraction_years=0.80,
    total_years=22,
):
    """
    Build a stable-footprint cropland mask (Option C).

    Criteria:
      - Pixel classified as cropland in ALL baseline years (e.g. 2001-2003)
      - Pixel classified as cropland in >= min_fraction_years of all years

    Args:
        annual_cropland_mask: ee.Image with bands 'lc_2001'..'lc_2022',
            each 1 where cropland, masked elsewhere.
        baseline_years: list of int (default [2001, 2002, 2003])
        min_fraction_years: float, fraction threshold (default 0.80)
        total_years: int, total number of years (default 22)

    Returns:
        ee.Image single-band binary mask ('stable_cropland'), 1 = stable cropland.
    """
    if baseline_years is None:
        baseline_years = [2001, 2002, 2003]

    # 1) Baseline check: must be cropland in ALL baseline years
    baseline_bands = [f'lc_{y}' for y in baseline_years]
    # unmask to 0 so min() works (min == 1 iff all are 1)
    baseline_stack = annual_cropland_mask.select(baseline_bands).unmask(0)
    baseline_all = baseline_stack.reduce(ee.Reducer.min())

    # 2) Frequency check: cropland in >= threshold of all years
    min_years = math.ceil(min_fraction_years * total_years)
    all_years_stack = annual_cropland_mask.unmask(0)
    cropland_count = all_years_stack.reduce(ee.Reducer.sum())
    freq_mask = cropland_count.gte(min_years)

    # 3) Combine: baseline AND frequency
    stable_mask = (
        baseline_all.And(freq_mask).rename('stable_cropland').selfMask()
    )

    return stable_mask


In [ ]:
# Cell 4: Main Processing Function

def process_export_a_tile(
    tile,
    filename_append,
    temporal_res='8day',
    cropland_mask_mode='both',
    debug=False,
):
    """
    Process and export annual max EVI for one tile.

    Args:
        tile: ee.Feature with tileID property
        filename_append: str, suffix for export filename
        temporal_res: '8day' or '32day'
        cropland_mask_mode: 'annual', 'stable', or 'both'
        debug: bool, print debug info
    """
    # Determine which masks to produce
    if cropland_mask_mode == 'both':
        do_annual = True
        do_stable = True
    elif cropland_mask_mode == 'annual':
        do_annual = True
        do_stable = False
    elif cropland_mask_mode == 'stable':
        do_annual = False
        do_stable = True
    else:
        raise ValueError(f"Unknown cropland_mask_mode: {cropland_mask_mode}")

    aoi = tile.geometry()
    tile_id = tile.get('tileID').getInfo()
    aoiBuffered = aoi.buffer(10000)

    def clip_(image):
        return image.clip(aoiBuffered)

    def filterBounds_(imageColl):
        return imageColl.filterBounds(aoiBuffered)

    # ------------------------------------------------------------------
    # Cropland mask (GLC-FCS30D annual), keep bands 2001-2022
    # ------------------------------------------------------------------
    glc30 = ee.ImageCollection(
        "projects/sat-io/open-datasets/GLC-FCS30D/annual"
    )
    glc30 = filterBounds_(glc30).mosaic()
    glc_band_years = ee.List.sequence(2000, 2022)
    new_names = glc_band_years.map(
        lambda y: ee.String('lc_').cat(ee.Number(y).format('%d'))
    )
    glc30 = glc30.rename(new_names)
    bands_to_keep = ee.List.sequence(2001, 2022).map(
        lambda y: ee.String('lc_').cat(ee.Number(y).format('%d'))
    )
    glc30 = glc30.select(bands_to_keep)

    # Build annual cropland mask
    annual_cropland_mask = build_annual_cropland_mask(glc30)
    annual_cropland_mask = clip_(annual_cropland_mask)

    if debug:
        print(
            f"Annual cropland mask bands: "
            f"{annual_cropland_mask.bandNames().size().getInfo()}"
        )

    # Build stable cropland mask
    stable_cropland_mask = None
    if do_stable:
        stable_cropland_mask = build_stable_cropland_mask(
            annual_cropland_mask,
            baseline_years=params.get('baseline_years', [2001, 2002, 2003]),
            min_fraction_years=params.get('min_fraction_years', 0.80),
            total_years=22,
        )
        stable_cropland_mask = clip_(stable_cropland_mask)
        if debug:
            print("Stable cropland mask built successfully.")

    # For grid generation, use ever-cropland mask (union across all years)
    ever_cropland_mask = annual_cropland_mask.reduce(ee.Reducer.max())

    # ------------------------------------------------------------------
    # Load pre-computed EVI composites
    # ------------------------------------------------------------------
    if temporal_res == '8day':
        evi_composite = ee.ImageCollection(
            "LANDSAT/COMPOSITES/C02/T1_L2_8DAY_EVI"
        )
    elif temporal_res == '32day':
        evi_composite = ee.ImageCollection(
            "LANDSAT/COMPOSITES/C02/T1_L2_32DAY_EVI"
        )
    else:
        raise ValueError(f"Unknown temporal_res: {temporal_res}")

    evi_composite = filterBounds_(evi_composite)

    # ------------------------------------------------------------------
    # Compute annual max EVI for each year (unmasked)
    # ------------------------------------------------------------------
    years = list(range(2001, 2023))

    annual_max_bands_raw = []
    for y in years:
        band = safe_annual_max(evi_composite, y, band='EVI')
        band = clip_(band)
        annual_max_bands_raw.append(band)

    # ------------------------------------------------------------------
    # Apply cropland masks and build stacks
    # ------------------------------------------------------------------

    # --- Annual mask: year-specific ---
    evi_max_annual_mask = None
    if do_annual:
        annual_masked = []
        for i, y in enumerate(years):
            year_mask = annual_cropland_mask.select(f'lc_{y}')
            masked_band = annual_max_bands_raw[i].updateMask(year_mask)
            annual_masked.append(masked_band)
        evi_max_annual_mask = ee.Image.cat(annual_masked)

    # --- Stable mask: single mask for all years ---
    evi_max_stable_mask = None
    if do_stable:
        stable_masked = []
        for i, y in enumerate(years):
            masked_band = annual_max_bands_raw[i].updateMask(
                stable_cropland_mask
            )
            stable_masked.append(masked_band)
        evi_max_stable_mask = ee.Image.cat(stable_masked)

    if debug:
        if evi_max_annual_mask is not None:
            print(
                f"Annual-mask stack bands: "
                f"{evi_max_annual_mask.bandNames().getInfo()}"
            )
        if evi_max_stable_mask is not None:
            print(
                f"Stable-mask stack bands: "
                f"{evi_max_stable_mask.bandNames().getInfo()}"
            )

    # ------------------------------------------------------------------
    # Grid generation (cropland threshold 1%)
    # ------------------------------------------------------------------
    grid_image = params['targetGrid']
    proj = grid_image.projection()
    coords = ee.Image.pixelCoordinates(proj)
    uniqueId = (
        coords.select('x')
        .multiply(1000000)
        .add(coords.select('y'))
        .toLong()
        .rename('cell_id')
    )
    grid_image = grid_image.addBands([uniqueId])

    grid_full = grid_image.select('cell_id').reduceToVectors(
        geometry=aoi,
        scale=10000,
        geometryType='polygon',
        labelProperty='cell_id',
    )

    cropFraction = ever_cropland_mask.unmask().reduceRegions(
        collection=grid_full,
        reducer=ee.Reducer.mean(),
        scale=100,
    )

    grid = cropFraction.filter(ee.Filter.gte('mean', 0.01))

    if debug:
        print(f"Grid cells (cropland >= 1%): {grid.size().getInfo()}")

    # ------------------------------------------------------------------
    # Reduce to grid & export (separate tasks per masking strategy)
    # ------------------------------------------------------------------
    started_tasks = []
    version = params['version']

    # Export with annual cropland mask
    if do_annual and evi_max_annual_mask is not None:
        grid_stack_annual = evi_max_annual_mask.reduceRegions(
            collection=grid,
            reducer=ee.Reducer.mean(),
            scale=30,
            tileScale=4,
            maxPixelsPerRegion=1e9,
        )
        task_name_annual = (
            f"{version}_evi_max_{temporal_res}"
            f"_annualMask{filename_append}"
        )
        task_annual = ee.batch.Export.table.toDrive(
            collection=grid_stack_annual,
            description=task_name_annual,
            folder=params['gdrive'],
            fileFormat='CSV',
        )
        task_annual.start()
        started_tasks.append(task_name_annual)

    # Export with stable cropland mask
    if do_stable and evi_max_stable_mask is not None:
        grid_stack_stable = evi_max_stable_mask.reduceRegions(
            collection=grid,
            reducer=ee.Reducer.mean(),
            scale=30,
            tileScale=4,
            maxPixelsPerRegion=1e9,
        )
        task_name_stable = (
            f"{version}_evi_max_{temporal_res}"
            f"_stableMask{filename_append}"
        )
        task_stable = ee.batch.Export.table.toDrive(
            collection=grid_stack_stable,
            description=task_name_stable,
            folder=params['gdrive'],
            fileFormat='CSV',
        )
        task_stable.start()
        started_tasks.append(task_name_stable)

    if debug:
        print(
            f"Started {len(started_tasks)} export task(s) for tile {tile_id}:"
        )
        for i, name in enumerate(started_tasks, 1):
            print(f"  {i}. {name}")


In [ ]:
# Cell 5: Test Execution (single tile, with 8-day -> 32-day fallback)

params['notRun'] = True  # Set to False to run test

test_tile_id = 63  # tile with lots of cropland; also try 152
temporal_res = params['temporal_res']  # '8day' or '32day'
mask_mode = params['cropland_mask_mode']  # 'annual', 'stable', or 'both'

if not params['notRun']:
    test_tile = tiles.filter(ee.Filter.eq('tileID', test_tile_id)).first()
    filename = f"__v_{params['version']}__test_t{test_tile_id}"

    print(
        f"Testing with tile {test_tile_id}, "
        f"temporal_res={temporal_res}, mask_mode={mask_mode}"
    )

    # Try preferred temporal_res first, fallback to 32-day
    try:
        print(f"Attempting with {temporal_res} composites...")
        process_export_a_tile(
            test_tile,
            filename,
            temporal_res=temporal_res,
            cropland_mask_mode=mask_mode,
            debug=True,
        )
        print(f"Successfully submitted tasks with {temporal_res} composites.")
    except Exception as e:
        print(f"Failed with {temporal_res}: {e}")
        if temporal_res == '8day':
            print("Retrying with 32-day composites...")
            try:
                process_export_a_tile(
                    test_tile,
                    filename,
                    temporal_res='32day',
                    cropland_mask_mode=mask_mode,
                    debug=True,
                )
                print("Successfully submitted tasks with 32-day composites.")
            except Exception as e2:
                print("Both attempts FAILED!")
                print(f"  8-day error: {e}")
                print(f"  32-day error: {e2}")
else:
    print("Test not run (params['notRun'] = True). Set to False to run.")


In [ ]:
# Cell 6: Full Execution
#
# Options:
#   - Run for ALL tiles
#   - Run for FIRST HALF of tiles
#   - Run for SECOND HALF of tiles
#
# Includes 8-day -> 32-day fallback per tile.

params['notRun'] = True  # Set to False to run

# Which subset to run: 'all', 'first_half', 'second_half'
run_subset = 'all'

temporal_res = params['temporal_res']
mask_mode = params['cropland_mask_mode']

# Tiles to skip (known problematic tiles)
skip_ids = {621, 620, 615, 614, 294, 293, 285}

if not params['notRun']:
    tile_ids_client = tiles.aggregate_array('tileID').getInfo()
    n_tiles = len(tile_ids_client)
    half = n_tiles // 2

    if run_subset == 'all':
        tile_range = range(n_tiles)
    elif run_subset == 'first_half':
        tile_range = range(half)
    elif run_subset == 'second_half':
        tile_range = range(half, n_tiles)
    else:
        raise ValueError(f"Unknown run_subset: {run_subset}")

    print(
        f"Running {run_subset}: {len(tile_range)} tiles, "
        f"temporal_res={temporal_res}, mask_mode={mask_mode}"
    )

    for i in tile_range:
        tile_id = tile_ids_client[i]

        if tile_id in skip_ids:
            print(f"Skipping tile: {tile_id}")
            continue

        tile = tiles.filter(ee.Filter.eq('tileID', tile_id)).first()
        filename = f"__v_{params['version']}__t_{tile_id}"

        # Try preferred temporal_res, fallback to 32-day
        try:
            process_export_a_tile(
                tile,
                filename,
                temporal_res=temporal_res,
                cropland_mask_mode=mask_mode,
                debug=False,
            )
            print(f"Tile {tile_id}: submitted with {temporal_res}")
        except Exception as e:
            print(f"Tile {tile_id} failed with {temporal_res}: {e}")
            if temporal_res == '8day':
                print(f"  Retrying tile {tile_id} with 32-day...")
                try:
                    process_export_a_tile(
                        tile,
                        filename,
                        temporal_res='32day',
                        cropland_mask_mode=mask_mode,
                        debug=False,
                    )
                    print(f"  Tile {tile_id}: submitted with 32-day")
                except Exception as e2:
                    print(f"  Tile {tile_id} FAILED both: {e2}")
else:
    print(
        "Full run not started (params['notRun'] = True). "
        "Set to False to run."
    )
